In [1]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  IN_COLAB = True
  work_dir = '/content/drive/MyDrive/COMP720Project/SampledExplanation'
except:
  IN_COLAB = False
  work_dir = input()

Mounted at /content/drive


In [ ]:
# !mkdir /content/drive/MyDrive/COMP720Project/SampledExplanation
# !cp /content/drive/MyDrive/COMP720Project/hbf/explanation_res.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_hbf.csv
# !cp /content/drive/MyDrive/COMP720Project/cf/temp/explanations.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_cf.csv
# !cp /content/drive/MyDrive/COMP720Project/rcbf_all_features/temp/explanations.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_cbf.csv

In [2]:
import os
os.chdir(work_dir)
os.listdir(work_dir)

['explanations_cf.csv',
 'explanations_cbf.csv',
 'explanations_hbf.csv',
 'score_distributions.csv',
 'scored_hbf.csv',
 'summary_means.csv',
 'scored_cbf.csv',
 'scored_cf.csv',
 'llm_plausibility_sample.csv']

In [3]:
import re
import numpy as np
import pandas as pd

SAMPLE_PERS = 100
RANDOM_SEED = 42


In [4]:
pd.set_option('max_colwidth', 512)
pd.set_option('display.max_rows', 500)


In [5]:
FILES = {
    "cbf": ("explanations_cbf.csv", "Content-Based"),
    "cf": ("explanations_cf.csv", "Collaborative"),
    "hbf": ("explanations_hbf.csv", "Hybrid"),
}

In [6]:
################
### Patterns ###
################

PAT_WATCH = re.compile(
    r'Because you watched "(?P<src>.*?)", which (?P<clauses>.*?), '
    r'we think you\'ll like "(?P<rec>.*?)"\.'
)
PAT_NO_HISTORY = re.compile(
    r'"(?P<rec>.*?)" is recommended based on your overall viewing patterns\.'
)
PAT_CF_ITEMS = re.compile(
    r'Because you enjoyed (?P<items>.*?), we think you\'ll like "(?P<rec>.*?)"\.'
)
PAT_CF_GENERIC = re.compile(
    r'"(?P<rec>.*?)" is broadly similar to your recent viewing\.'
)
ITEM_PAT = re.compile(r'(?P<title>.*?) \(your rating: (?P<rating>[\d.]+)\)')

CLAUSE_GENRE = re.compile(r'shares the (?P<genres>[^()]*) genre\(s\)')
CLAUSE_THEME = "has a similar theme/plot"
# Text says "story elements", but the generator fires this clause off keyword_sim
# (keyword-embedding similarity), not a literal plot/story comparison -- named for
# the underlying signal rather than the surface wording, to match CLAUSE_RATED_HISTORY
# below (also named for its signal, collab_sim, rather than its wording).
CLAUSE_KEYWORD = "touches on similar story elements"
# "Taste" here is really the model's learned collaborative-filtering item embedding
# (collab_sim), trained on the interaction/rating history -- the same kind of signal
# CF's "items"/"generic" templates are built on, hence the shared name.
CLAUSE_RATED_HISTORY = "is often watched by users with similar taste to yours"
CLAUSE_STYLE = "is broadly similar in style"

# genre/theme/keyword come from content similarity (genre overlap, description
# embedding, keyword embedding); rated_item_history comes from collab_sim -- the
# same kind of signal CF's "items"/"generic" templates are built on (see parse_cf
# below and explanations.py). A plausibility judge grading content-based
# credibility should be expected to treat these the same way regardless of which
# file they came from -- signal_type() below groups them for that comparison.
def signal_type(key, template):
    if template in ("unparsed", "no_history", "style_generic"):
        return template
    if "rated_item_history" in template:
        return "collaborative"
    if key == "cf":
        return "collaborative"
    return "content"

In [7]:
def parse_cbf_hbf(exp):
    m = PAT_WATCH.match(exp)
    if m:
        src, rec = m.group("src").strip(), m.group("rec").strip()
        clauses = [c.strip() for c in m.group("clauses").split(" and ")]

        genres = []
        theme = keyword = rated_item_history = False
        recognized = 0
        for clause in clauses:
            gm = CLAUSE_GENRE.fullmatch(clause)
            if gm:
                genres = [g.strip() for g in gm.group("genres").split(",") if g.strip()]
                recognized += 1
            elif clause == CLAUSE_THEME:
                theme = True
                recognized += 1
            elif clause == CLAUSE_KEYWORD:
                keyword = True
                recognized += 1
            elif clause == CLAUSE_RATED_HISTORY:
                rated_item_history = True
                recognized += 1
            # else: unrecognized clause text (e.g. the CLAUSE_STYLE fallback,
            # which only ever appears alone) -- falls through to style_generic below.

        if genres:
            template = "genre"
        elif recognized == 0:
            # No genre and nothing else recognized either -- covers CLAUSE_STYLE
            # ("is broadly similar in style") and any unrecognized clause text.
            template = "style_generic"
        else:
            template = "+".join(name for name, present in
                                 (("theme", theme), ("keyword", keyword),
                                  ("rated_item_history", rated_item_history)) if present)

        return {"template": template, "src": src, "rec": rec,
                "n_genres": len(genres), "theme": theme, "keyword": keyword,
                "rated_item_history": rated_item_history}

    m = PAT_NO_HISTORY.match(exp)
    if m:
        return {"template": "no_history", "src": None, "rec": m.group("rec").strip(),
                "n_genres": 0, "theme": False, "keyword": False, "rated_item_history": False}

    return {"template": "unparsed", "src": None, "rec": None,
            "n_genres": 0, "theme": False, "keyword": False, "rated_item_history": False}



In [8]:
def parse_cf(exp):
    m = PAT_CF_ITEMS.match(exp)
    if m:
        items_str = m.group("items")
        parts = re.split(r"(?<=\)), ", items_str)
        titles, ratings = [], []
        for p in parts:
            im = ITEM_PAT.match(p.strip())
            if im:
                titles.append(im.group("title").strip())
                ratings.append(float(im.group("rating")))
        return {"template": "items", "rec": m.group("rec").strip(), "titles": titles,
                "ratings": ratings, "n_items": len(titles)}
    m = PAT_CF_GENERIC.match(exp)
    if m:
        return {"template": "generic", "rec": m.group("rec").strip(), "titles": [], "ratings": [], "n_items": 0}
    return {"template": "unparsed", "rec": None, "titles": [], "ratings": [], "n_items": 0}




In [9]:

results = {}

for key, (fname, label) in FILES.items():
    df = pd.read_csv(fname)

    rows = []
    for _, row in df.iterrows():
        exp = row["explanation"]
        if key == "cf":
            feat = parse_cf(exp)
            n_evidence = feat["n_items"]
            self_rec = feat["rec"] is not None and feat["rec"].strip().lower() in [t.lower() for t in feat["titles"]]
        else:
            feat = parse_cbf_hbf(exp)
            n_evidence = feat["n_genres"]
            self_rec = (feat["src"] is not None and feat["rec"] is not None
                        and feat["src"].strip().lower() == feat["rec"].strip().lower())

        rows.append({
            "user_id": row["user_id"], "item_id": row["item_id"], "explanation": exp,
            "template": feat["template"], "signal": signal_type(key, feat["template"]),
            "n_evidence": n_evidence, "self_rec_bug": self_rec,
        })
    parsed = pd.DataFrame(rows)
    parsed.to_csv(f"scored_{key}.csv", index=False)
    results[key] = (label, parsed)

# ---- Parsing report ----
print("=" * 70)
print("Parsed the full explanation file per method (no sampling)")
print("=" * 70)

Parsed the full explanation file per method (no sampling)


In [10]:
summary_rows = []
for key, (label, parsed) in results.items():
    self_rec_rate = parsed["self_rec_bug"].mean()
    template_dist = parsed["template"].value_counts(normalize=True).round(3).to_dict()
    unparsed_count = int((parsed["template"] == "unparsed").sum())
    print(f"\n--- {label} ---")
    print(f"  Unparsed count: {unparsed_count} / {len(parsed)}")
    print(f"  Self-recommendation bug rate: {self_rec_rate:.1%}")
    print(f"  Template mix: {template_dist}")
    summary_rows.append({
        "method": label, "file": key,
        "unparsed_count": unparsed_count,
        "self_rec_bug_rate": round(self_rec_rate, 3),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("summary_means.csv", index=False)
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
display(summary_df)


--- Content-Based ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'genre': 0.974, 'style_generic': 0.019, 'keyword': 0.007}

--- Collaborative ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'items': 0.553, 'generic': 0.447}

--- Hybrid ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'genre': 0.945, 'rated_item_history': 0.042, 'style_generic': 0.007, 'keyword': 0.003, 'keyword+rated_item_history': 0.002, 'theme+rated_item_history': 0.001}

SUMMARY TABLE


,method,file,unparsed_count,self_rec_bug_rate
0,Content-Based,cbf,0,0.0
1,Collaborative,cf,0,0.0
2,Hybrid,hbf,0,0.0


## LLM plausibility judge (sample 50 items)

In [11]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.6 MB/s eta 0:00:00


In [12]:
import os
import time

import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

client = anthropic.Anthropic()

JUDGE_MODEL = "claude-sonnet-5"

JUDGE_PROMPT = """You will be shown one movie recommendation explanation at a time. Each states a reason for recommending a film based on something the user previously watched or rated.
Using your knowledge of the films named, judge whether the stated reason is credible: would a viewer who knows these films find this a sensible basis for the recommendation, or would the connection seem arbitrary?
Rate plausibility from 1 to 5, where 1 means the cited films have no meaningful relationship to the recommendation and 5 means the connection is clear and well-founded. Judge only what the sentence actually argues -- do not credit a recommendation that happens to be good if the stated reason does not support it.
Give the rating and one sentence of justification naming the specific films. Do not consider how much evidence is cited; a reason citing one film may be more credible than one citing three.
You have a web_search tool. Use it to confirm plot, genre, or thematic details of the named films whenever you are not fully confident from memory alone -- the judgment should reflect what the films are actually about, not a guess.
Respond with nothing but the rating and justification, in exactly this format:
Rating: <integer 1-5>
Justification: <one sentence, naming the specific films>"""

LLM_SAMPLE_N = 50   # rows per file
LLM_RNG_SEED = 42

Anthropic API key: ··········


In [13]:
RATING_PAT = re.compile(r"Rating:\s*([1-5]).*?Justification:\s*(.+)", re.DOTALL)

def judge_plausibility(explanation):
    try:
        response = client.messages.create(
            model=JUDGE_MODEL,
            max_tokens=8192,
            system=JUDGE_PROMPT,
            output_config={"effort": "medium"},
            tools=[{"type": "web_search_20260209", "name": "web_search", "max_uses": 1}],
            messages=[{"role": "user", "content": explanation}],
        )
    except anthropic.APIError as e:
        return {"rating": None, "justification": f"API error: {e}"}

    text = " ".join(b.text for b in response.content if b.type == "text").strip()
    m = RATING_PAT.search(text)
    if m:
        return {"rating": int(m.group(1)), "justification": m.group(2).strip()}
    return {"rating": None, "justification": text or None}

In [14]:
llm_rows = []

for key, (label, parsed) in results.items():
    sample = parsed.sample(n=min(LLM_SAMPLE_N, len(parsed)), random_state=LLM_RNG_SEED)
    for i, (_, row) in enumerate(sample.iterrows()):
        judged = judge_plausibility(row["explanation"])
        llm_rows.append({
            "method": label, "file": key,
            "user_id": row["user_id"], "item_id": row["item_id"],
            "explanation": row["explanation"], "template": row["template"], "signal": row["signal"],
            "llm_plausibility": judged["rating"], "llm_justification": judged["justification"],
        })
        print(f"[{label}] {i + 1}/{len(sample)}  rating={judged['rating']}")

llm_df = pd.DataFrame(llm_rows)
llm_df.to_csv("llm_plausibility_sample.csv", index=False)

[Content-Based] 1/50  rating=3
[Content-Based] 2/50  rating=3
[Content-Based] 3/50  rating=3
[Content-Based] 4/50  rating=5
[Content-Based] 5/50  rating=5
[Content-Based] 6/50  rating=4
[Content-Based] 7/50  rating=4
[Content-Based] 8/50  rating=5
[Content-Based] 9/50  rating=2
[Content-Based] 10/50  rating=5
[Content-Based] 11/50  rating=4
[Content-Based] 12/50  rating=3
[Content-Based] 13/50  rating=2
[Content-Based] 14/50  rating=4
[Content-Based] 15/50  rating=3
[Content-Based] 16/50  rating=4
[Content-Based] 17/50  rating=2
[Content-Based] 18/50  rating=5
[Content-Based] 19/50  rating=2
[Content-Based] 20/50  rating=3
[Content-Based] 21/50  rating=4
[Content-Based] 22/50  rating=2
[Content-Based] 23/50  rating=4
[Content-Based] 24/50  rating=3
[Content-Based] 25/50  rating=1
[Content-Based] 26/50  rating=2
[Content-Based] 27/50  rating=5
[Content-Based] 28/50  rating=4
[Content-Based] 29/50  rating=4
[Content-Based] 30/50  rating=2
[Content-Based] 31/50  rating=5
[Content-Based] 3

In [20]:
# llm_df

In [16]:
print("=" * 70)
print(f"LLM plausibility sample: {LLM_SAMPLE_N} rows/file (seed={LLM_RNG_SEED})")
print("=" * 70)

for method, group in llm_df.groupby("method"):
    rated = group["llm_plausibility"].dropna()
    print(f"\n--- {method} ---")
    print(f"  Rated: {len(rated)} / {len(group)}")
    if len(rated):
        print(f"  Mean LLM plausibility: {rated.mean():.2f}")

print("\n" + "=" * 70)
print("BY SIGNAL TYPE (pooled across files)")
print("=" * 70)

for signal, group in llm_df.groupby("signal"):
    rated = group["llm_plausibility"].dropna()
    print(f"\n--- {signal} ---")
    print(f"  Rated: {len(rated)} / {len(group)}")
    if len(rated):
        print(f"  Mean LLM plausibility: {rated.mean():.2f}")

display(llm_df[["method", "user_id", "item_id", "template", "signal", "llm_plausibility", "llm_justification"]])

LLM plausibility sample: 50 rows/file (seed=42)

--- Collaborative ---
  Rated: 50 / 50
  Mean LLM plausibility: 1.64

--- Content-Based ---
  Rated: 50 / 50
  Mean LLM plausibility: 3.28

--- Hybrid ---
  Rated: 50 / 50
  Mean LLM plausibility: 2.92

BY SIGNAL TYPE (pooled across files)

--- collaborative ---
  Rated: 53 / 53
  Mean LLM plausibility: 1.62

--- content ---
  Rated: 96 / 96
  Mean LLM plausibility: 3.17

--- style_generic ---
  Rated: 1 / 1
  Mean LLM plausibility: 2.00


,method,user_id,item_id,template,signal,llm_plausibility,llm_justification
0,Content-Based,26,875,genre,content,3,"Both ""Harry Potter and the Deathly Hallows: Part 2"" and ""Hellboy"" are fantasy films with supernatural battles between good and evil, but their tone, style, and target audience differ enough (epic YA wizarding saga vs. dark comic-book horror-action) that genre alone is a fairly loose basis for the recommendation."
1,Content-Based,36,427,genre,content,3,"""A Beautiful Mind"" and ""To Kill a Mockingbird"" are both prestige Dramas with themes of moral/psychological struggle, but their genre overlap is generic since Drama is an extremely broad category, making this a plausible but weak, largely superficial connection rather than a substantive thematic match."
2,Content-Based,37,7835,genre,content,3,"""Iron Man"" and ""Inception"" do share broad Action/Adventure/Sci-Fi genre classification, but beyond that surface overlap—superhero blockbuster vs. cerebral heist/dream-thriller—the films differ substantially in tone, themes, and style, making the connection plausible but fairly generic."
3,Content-Based,33,86,genre,content,5,"Both ""Howl's Moving Castle"" and ""Princess Mononoke"" are Studio Ghibli films directed by Hayao Miyazaki that blend Adventure, Animation, and Fantasy genres with shared thematic elements like nature, war, and transformation/curses, making this a strong and credible connection."
4,Content-Based,20,4122,genre,content,5,"""A Nightmare on Elm Street"" and ""Wes Craven's New Nightmare"" are both directed by Wes Craven, share the Freddy Krueger character and dream-invasion horror premise, with the latter being a meta-sequel directly referencing the former, making the genre and story-element connection well-founded."
5,Content-Based,33,82,genre,content,4,"""Howl's Moving Castle"" and ""The Lord of the Rings: The Return of the King"" are both fantasy-adventure epics involving magic, quests, and battles against dark forces, so the shared-genre reasoning is sensible even if the two films differ notably in tone and target audience."
6,Content-Based,31,511,genre,content,4,"Both ""Leaving Las Vegas"" and ""Dead Man Walking"" are serious 1995 Drama films dealing with weighty personal/moral crises, so the genre-based connection is reasonable, though it rests only on shared genre rather than deeper thematic overlap like addiction versus capital punishment."
7,Content-Based,25,68,genre,content,5,"Both ""Reservoir Dogs"" and ""Snatch"" are stylized crime films built around interconnected criminal schemes, sharp dialogue, dark humor, and ensemble casts of tough-talking crooks, making the genre/thematic connection well-founded."
8,Content-Based,42,1217,genre,content,2,"""Menace II Society"" is a gritty urban crime drama about gang violence in South Central LA, whereas ""Nick of Time"" is a real-time political-assassination thriller centered on an ordinary man coerced into a plot—so while both involve ""crime"" loosely, their tone, setting, and story elements are quite different, making the connection tenuous."
9,Content-Based,6,3492,genre,content,5,"Both ""Sleepless in Seattle"" and ""It Could Happen to You"" are 1990s romantic dramedies starring Nicolas Cage-era/Meg Ryan-adjacent leads (with Ryan actually starring in the latter) built around fate-driven, feel-good romance, making the shared Comedy/Drama/Romance genre connection well-founded."


In [19]:
llm_df.to_csv("llm_plausibility_sample.csv", index=False)